In [2]:
import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime

# Project paths
RAW_DATA = Path("../01_Raw_Data/marketing_data.xlsx")
OUTPUT_DIR = Path("../05_Clean_Data")

# Output qovluğunu yaradırıq
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Raw data
df_raw = pd.read_excel(RAW_DATA)

# Working copy
df = df_raw.copy()

print("Raw Data Shape:", df_raw.shape)

display(df.head())

Raw Data Shape: (2240, 28)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Response,Complain,Country
0,1826,1970,Graduation,Divorced,84835.0,0,0,2014-06-16,0,189,...,6,1,0,0,0,0,0,1,0,Spain
1,1,1961,Graduation,Single,57091.0,0,0,2014-06-15,0,464,...,7,5,0,0,0,0,1,1,0,Canada
2,10476,1958,Graduation,Married,67267.0,0,1,2014-05-13,0,134,...,5,2,0,0,0,0,0,0,0,USA
3,1386,1967,Graduation,Together,32474.0,1,1,2014-05-11,0,10,...,2,7,0,0,0,0,0,0,0,Australia
4,5371,1989,Graduation,Single,21474.0,1,0,2014-04-08,0,6,...,2,7,1,0,0,0,0,1,0,Spain


In [10]:
df["Total_Purchases"] = (
    df["NumWebPurchases"]
    + df["NumCatalogPurchases"]
    + df["NumStorePurchases"]
)

spending_columns = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds"
]

df["Total_Spending"] = df[spending_columns].sum(axis=1)

In [3]:
cleaning_logs = []

def add_log(step, column, issue, action, affected_rows):

    cleaning_logs.append({
        "Step": step,
        "Column": column,
        "Issue": issue,
        "Action": action,
        "Affected_Rows": affected_rows,
        "Timestamp": datetime.now()
    })

In [11]:
# DQ-004: Endirimli alış sayı kanal alışlarının cəmindən böyükdür

df["Deal_Purchase_Issue"] = (
    df["NumDealsPurchases"] > df["Total_Purchases"]
).astype(int)

dq004_count = df["Deal_Purchase_Issue"].sum()

print("DQ-004 Affected Customers:", dq004_count)

DQ-004 Affected Customers: 3


In [12]:
# DQ-005: Alış sayı sıfırdır, amma xərcləmə müsbətdir

df["Zero_Purchase_Spending_Issue"] = (
    (df["Total_Purchases"] == 0) &
    (df["Total_Spending"] > 0)
).astype(int)

dq005_count = df["Zero_Purchase_Spending_Issue"].sum()

print("DQ-005 Affected Customers:", dq005_count)

DQ-005 Affected Customers: 6


In [13]:
df["Has_Data_Quality_Issue"] = (
    (df["Deal_Purchase_Issue"] == 1) |
    (df["Zero_Purchase_Spending_Issue"] == 1)
).astype(int)

df["Data_Quality_Status"] = np.where(
    df["Has_Data_Quality_Issue"] == 1,
    "Needs Review",
    "No Flagged Issue"
)

display(df["Data_Quality_Status"].value_counts())

Data_Quality_Status
No Flagged Issue    2233
Needs Review           7
Name: count, dtype: int64

In [18]:
# Sütun adlarındakı artıq boşluqları silirik
df.columns = df.columns.str.strip()

# Income sütununun mövcudluğunu yoxlayırıq
print("Income" in df.columns)

df["Income_Imputed"] = (
    df["Income"].fillna(df["Income"].median())
)

True


In [19]:
# Mövcud Data Quality yoxlamaları
quality_report = pd.DataFrame({
    "Quality_Check": [
        "Duplicate Customer IDs",
        "Missing Customer IDs",
        "Missing Income (Original)",
        "Missing Income (Imputed)",
        "Negative Total Spending",
        "Negative Total Purchases"
    ],
    "Issue_Count": [
        int(df["ID"].duplicated().sum()),
        int(df["ID"].isna().sum()),
        int(df["Income"].isna().sum()),
        int(df["Income_Imputed"].isna().sum()),
        int((df["Total_Spending"] < 0).sum()),
        int((df["Total_Purchases"] < 0).sum())
    ]
})

# DQ-004 və DQ-005 yoxlamaları
new_quality_checks = pd.DataFrame({
    "Quality_Check": [
        "DQ-004: Deal Purchases > Total Purchases",
        "DQ-005: Zero Purchases with Positive Spending",
        "Unique Customers with Flagged Issues"
    ],
    "Issue_Count": [
        int(df["Deal_Purchase_Issue"].sum()),
        int(df["Zero_Purchase_Spending_Issue"].sum()),
        int(df["Has_Data_Quality_Issue"].sum())
    ]
})

# Bütün yoxlamaları birləşdiririk
quality_report = pd.concat(
    [quality_report, new_quality_checks],
    ignore_index=True
)

display(quality_report)

,Quality_Check,Issue_Count
0,Duplicate Customer IDs,0
1,Missing Customer IDs,0
2,Missing Income (Original),24
3,Missing Income (Imputed),0
4,Negative Total Spending,0
5,Negative Total Purchases,0
6,DQ-004: Deal Purchases > Total Purchases,3
7,DQ-005: Zero Purchases with Positive Spending,6
8,Unique Customers with Flagged Issues,7


In [20]:
display(quality_report)

,Quality_Check,Issue_Count
0,Duplicate Customer IDs,0
1,Missing Customer IDs,0
2,Missing Income (Original),24
3,Missing Income (Imputed),0
4,Negative Total Spending,0
5,Negative Total Purchases,0
6,DQ-004: Deal Purchases > Total Purchases,3
7,DQ-005: Zero Purchases with Positive Spending,6
8,Unique Customers with Flagged Issues,7


In [21]:
cleaning_log_df = pd.DataFrame([
    {
        "Step": "01",
        "Column": "Column Names",
        "Action": "Remove extra spaces",
        "Method": "str.strip()"
    },
    {
        "Step": "02",
        "Column": "Income",
        "Action": "Handle missing values",
        "Method": "Median Imputation"
    },
    {
        "Step": "03",
        "Column": "ID",
        "Action": "Check duplicate records",
        "Method": "duplicated()"
    },
    {
        "Step": "04",
        "Column": "Categorical Columns",
        "Action": "Standardize categories",
        "Method": "replace() / map()"
    },
    {
        "Step": "05",
        "Column": "Income",
        "Action": "Detect outliers",
        "Method": "IQR"
    },
    {
        "Step": "06",
        "Column": "Analytical Columns",
        "Action": "Create calculated columns",
        "Method": "Feature Engineering"
    },
    {
        "Step": "07",
        "Column": "NumDealsPurchases",
        "Action": "Flag DQ-004 inconsistencies",
        "Method": "Conditional Validation"
    },
    {
        "Step": "08",
        "Column": "Total_Purchases / Total_Spending",
        "Action": "Flag DQ-005 inconsistencies",
        "Method": "Conditional Validation"
    }
])

display(cleaning_log_df)

,Step,Column,Action,Method
0,01,Column Names,Remove extra spaces,str.strip()
1,02,Income,Handle missing values,Median Imputation
2,03,ID,Check duplicate records,duplicated()
3,04,Categorical Columns,Standardize categories,replace() / map()
4,05,Income,Detect outliers,IQR
5,06,Analytical Columns,Create calculated columns,Feature Engineering
6,07,NumDealsPurchases,Flag DQ-004 inconsistencies,Conditional Validation
7,08,Total_Purchases / Total_Spending,Flag DQ-005 inconsistencies,Conditional Validation


In [22]:
required_columns = [
    "Income",
    "Income_Imputed",
    "Total_Spending",
    "Total_Purchases",
    "Education_Standardized",
    "Marital_Status_Group",
    "Age_Group",
    "Income_Group",
    "Deal_Purchase_Issue",
    "Zero_Purchase_Spending_Issue",
    "Has_Data_Quality_Issue",
    "Data_Quality_Status"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("Missing Columns:", missing_columns)

Missing Columns: ['Education_Standardized', 'Marital_Status_Group', 'Age_Group', 'Income_Group']


In [23]:
# Education Standardization

education_mapping = {
    "Basic": "Basic Education",
    "2n Cycle": "Second Cycle",
    "Graduation": "Graduate",
    "Master": "Master's Degree",
    "PhD": "Doctorate"
}

df["Education_Standardized"] = (
    df["Education"]
    .astype("string")
    .str.strip()
    .replace(education_mapping)
)


# Marital Status Standardization

marital_mapping = {
    "Married": "Married",
    "Together": "Partnered",
    "Single": "Single",
    "Alone": "Living Alone",
    "Divorced": "Divorced",
    "Widow": "Widowed",
    "Absurd": "Unknown",
    "YOLO": "Unknown"
}

df["Marital_Status_Group"] = (
    df["Marital_Status"]
    .astype("string")
    .str.strip()
    .map(marital_mapping)
)

In [24]:
# Təxmini yaş
reference_year = 2014

df["Customer_Age_Estimated"] = (
    reference_year - pd.to_numeric(
        df["Year_Birth"],
        errors="coerce"
    )
)

# Yaş qrupları
age_bins = [18, 25, 35, 45, 55, 65, np.inf]

age_labels = [
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65+"
]

valid_age = df["Customer_Age_Estimated"].where(
    df["Customer_Age_Estimated"].between(18, 100)
)

df["Age_Group"] = pd.cut(
    valid_age,
    bins=age_bins,
    labels=age_labels,
    right=False
)

In [25]:
df["Income_Group"] = pd.qcut(
    df["Income"],
    q=4,
    labels=[
        "Low Income",
        "Lower-Middle Income",
        "Upper-Middle Income",
        "High Income"
    ]
)

In [26]:
required_columns = [
    "Income",
    "Income_Imputed",
    "Total_Spending",
    "Total_Purchases",
    "Education_Standardized",
    "Marital_Status_Group",
    "Age_Group",
    "Income_Group",
    "Deal_Purchase_Issue",
    "Zero_Purchase_Spending_Issue",
    "Has_Data_Quality_Issue",
    "Data_Quality_Status"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("Missing Columns:", missing_columns)

Missing Columns: []


In [27]:
display(
    df[
        [
            "ID",
            "Education_Standardized",
            "Marital_Status_Group",
            "Customer_Age_Estimated",
            "Age_Group",
            "Income_Group"
        ]
    ].head(10)
)

,ID,Education_Standardized,Marital_Status_Group,Customer_Age_Estimated,Age_Group,Income_Group
0,1826,Graduate,Divorced,44,35-44,High Income
1,1,Graduate,Single,53,45-54,Upper-Middle Income
2,10476,Graduate,Married,56,55-64,Upper-Middle Income
3,1386,Graduate,Partnered,47,45-54,Low Income
4,5371,Graduate,Single,25,25-34,Low Income
5,7348,Doctorate,Single,56,55-64,High Income
6,4073,Second Cycle,Married,60,55-64,Upper-Middle Income
7,1991,Graduate,Partnered,47,45-54,Lower-Middle Income
8,4047,Doctorate,Married,60,55-64,Upper-Middle Income
9,9477,Doctorate,Married,60,55-64,Upper-Middle Income


In [28]:
from pathlib import Path

output_dir = Path("../05_Clean_Data")

output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "marketing_data_clean.csv"
excel_path = output_dir / "marketing_data_clean.xlsx"

# Tableau üçün CSV
df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

# Excel formatı
df.to_excel(
    excel_path,
    index=False
)

print("Export Completed Successfully!")

Export Completed Successfully!


In [29]:
report_path = output_dir / "data_quality_report.xlsx"

with pd.ExcelWriter(
    report_path,
    engine="openpyxl"
) as writer:

    quality_report.to_excel(
        writer,
        sheet_name="Quality_Checks",
        index=False
    )

    cleaning_log_df.to_excel(
        writer,
        sheet_name="Cleaning_Log",
        index=False
    )

print("Data Quality Report Exported!")

Data Quality Report Exported!
